# Эксперимент 3: LightGBM с конвейерными фичами

**Гипотеза:** конвейерная природа `status_1 → status_2 → status_3 → target` даёт прямой сигнал:
`status_3` в момент cutoff (10:30) напрямую предсказывает `target` через 30 мин (11:00).
Добавление лаговых status-фичей из конвейера существенно улучшит прогноз, особенно для ближних горизонтов.

**Подход:** одна глобальная LightGBM модель, `horizon` (1–8) как фича (direct multi-step).

**Ключевые новшества:**
- `status_3_at_ref` — прямой конвейерный сигнал (лаг 30 мин до отгрузки подтверждён на 100% маршрутов)
- `status_2_at_ref`, `status_1_at_ref` — сигналы для дальних горизонтов
- Лаги и rolling-средние status_3, status_1/2
- Расширенные rolling-статистики таргета (2h / 6h / 12h / 24h / 7d)
- `pipeline_signal` — адаптивный: для h≤2 берём status_3, для h≤4 — status_2, иначе status_1

**Валидация:** rolling-origin, 3 субботних cutoff (Oct 11 / Oct 18 / Oct 25 в 10:30).
Метрика: **WAPE + |Relative Bias|**. Бейзлайн: **0.382**.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import time

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='whitegrid', palette='muted')

# ─── Пути ──────────────────────────────────────────────────────────────────
DATA_DIR = Path('/Users/melikhovartem/Desktop/ИЗИ 200к/Data')

# ─── Константы ─────────────────────────────────────────────────────────────
STEP     = pd.Timedelta('30min')
HORIZONS = list(range(1, 9))      # горизонты 1..8 (30 мин → 4 ч)

# ─── Точки обрезки: 3 субботы перед тестом (единые для всех экспериментов) ─
VAL_CUTOFFS = [
    pd.Timestamp('2025-10-11 10:30:00'),
    pd.Timestamp('2025-10-18 10:30:00'),
    pd.Timestamp('2025-10-25 10:30:00'),
]
TEST_CUTOFF = pd.Timestamp('2025-11-01 10:30:00')

# ─── Параметры LightGBM ────────────────────────────────────────────────────
LGBM_PARAMS = dict(
    objective='regression_l1',   # MAE — ближайший к WAPE
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

# Субсэмплинг reference timestamps для ускорения (каждый N-й шаг)
SUBSAMPLE_N = 4

print("Конфигурация загружена.")
print(f"  Горизонты: {HORIZONS} × 30мин = 4ч")
print(f"  Фолды: {[str(c.date()) for c in VAL_CUTOFFS]}")

In [ ]:
train = pd.read_parquet(DATA_DIR / 'train_solo_track.parquet')
test  = pd.read_parquet(DATA_DIR / 'test_solo_track.parquet')

train['timestamp'] = pd.to_datetime(train['timestamp'])
test['timestamp']  = pd.to_datetime(test['timestamp'])

train = train.sort_values(['route_id', 'timestamp']).reset_index(drop=True)
test  = test.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

print(f"Train: {train.shape}")
print(f"  от {train.timestamp.min()} до {train.timestamp.max()}")
print(f"Test:  {test.shape}")
print(f"  от {test.timestamp.min()} до {test.timestamp.max()}")
print(f"\nКолонки train: {list(train.columns)}")
print(f"Колонки test:  {list(test.columns)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Метрики — ЕДИНЫЕ ДЛЯ ВСЕХ ЭКСПЕРИМЕНТОВ
# ══════════════════════════════════════════════════════════════════════════════

def wape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Weighted Absolute Percentage Error."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum(np.abs(y_true - y_pred)) / np.sum(y_true))

def relative_bias(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Относительное смещение (signed)."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float((np.sum(y_pred) - np.sum(y_true)) / np.sum(y_true))

def combined_metric(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """WAPE + |Relative Bias| — основная метрика соревнования."""
    return wape(y_true, y_pred) + abs(relative_bias(y_true, y_pred))

BASELINE = 0.382  # лучший бейзлайн: среднее по маршруту
print(f"Метрики определены. Бейзлайн: {BASELINE}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Feature Engineering
# ══════════════════════════════════════════════════════════════════════════════

def compute_route_stats(data: pd.DataFrame) -> pd.DataFrame:
    """Маршрутные статистики из обучающих данных (до cutoff)."""
    stats = (
        data.groupby('route_id')['target_1h']
        .agg(
            route_mean='mean',
            route_median='median',
            route_std='std',
            route_q25=lambda x: x.quantile(0.25),
            route_q75=lambda x: x.quantile(0.75),
            route_zero_frac=lambda x: (x == 0).mean(),
        )
        .reset_index()
    )
    stats['route_cv'] = stats['route_std'] / (stats['route_mean'] + 1.0)
    return stats


def compute_ref_features(data: pd.DataFrame, route_stats: pd.DataFrame) -> pd.DataFrame:
    """
    Вычислить признаки для каждой строки (route_id, timestamp).

    Все lag/rolling используют только прошлые значения (.shift(1)) — нет утечки.
    data должна быть отсортирована по (route_id, timestamp).
    """
    data = data.sort_values(['route_id', 'timestamp']).copy()

    # ── Лаги таргета ──────────────────────────────────────────────────────────
    for lag in [1, 2, 3, 4, 6, 12, 24, 48, 96, 336]:
        data[f'target_lag_{lag}'] = data.groupby('route_id')['target_1h'].shift(lag)

    # ── Скользящие средние таргета (shift(1): текущий момент не включается) ───
    for window, name in [(4,'2h'), (12,'6h'), (24,'12h'), (48,'24h'), (336,'7d')]:
        data[f'target_roll_mean_{name}'] = data.groupby('route_id')['target_1h'].transform(
            lambda x, w=window: x.shift(1).rolling(w, min_periods=1).mean()
        )

    # ── Скользящие стандартные отклонения ─────────────────────────────────────
    for window, name in [(12,'6h'), (48,'24h')]:
        data[f'target_roll_std_{name}'] = data.groupby('route_id')['target_1h'].transform(
            lambda x, w=window: x.shift(1).rolling(w, min_periods=2).std()
        )

    # ── EWM (экспоненциально взвешенное среднее, span=4 = 2ч) ─────────────────
    data['target_ewm_span4'] = data.groupby('route_id')['target_1h'].transform(
        lambda x: x.shift(1).ewm(span=4, min_periods=1).mean()
    )

    # ── Разности ──────────────────────────────────────────────────────────────
    data['target_diff_1']  = data.groupby('route_id')['target_1h'].diff(1)
    data['target_diff_48'] = data.groupby('route_id')['target_1h'].diff(48)

    # ── Status-фичи (значение В момент reference = "at_ref") ──────────────────
    for s in ['status_1', 'status_2', 'status_3', 'status_4', 'status_5']:
        data[f'{s}_at_ref'] = data[s]

    # ── Лаги status_3 (конвейерный сигнал: шаги перед текущим) ───────────────
    for lag in [1, 2, 3, 4]:
        data[f'status_3_lag_{lag}'] = data.groupby('route_id')['status_3'].shift(lag)

    # ── Rolling mean status_1/2/3 за 2ч (4 шага) ─────────────────────────────
    for s in ['status_1', 'status_2', 'status_3']:
        data[f'{s}_roll_mean_2h'] = data.groupby('route_id')[s].transform(
            lambda x: x.shift(1).rolling(4, min_periods=1).mean()
        )

    # ── Rolling mean status_4/5 за 24ч (48 шагов) ────────────────────────────
    for s in ['status_4', 'status_5']:
        data[f'{s}_roll_mean_24h'] = data.groupby('route_id')[s].transform(
            lambda x: x.shift(1).rolling(48, min_periods=1).mean()
        )

    # ── Временные признаки точки отсчёта ─────────────────────────────────────
    data['ref_hour'] = data['timestamp'].dt.hour
    data['ref_dow']  = data['timestamp'].dt.dayofweek

    # ── Маршрутные статистики ─────────────────────────────────────────────────
    data = data.merge(route_stats, on='route_id', how='left')
    return data


def add_horizon_features(df: pd.DataFrame, h: int) -> pd.DataFrame:
    """Добавить признаки горизонта и целевого момента времени."""
    df = df.copy()
    df['horizon'] = h
    df['tgt_ts']  = df['timestamp'] + h * STEP

    tgt_hour = df['tgt_ts'].dt.hour + df['tgt_ts'].dt.minute / 60
    tgt_dow  = df['tgt_ts'].dt.dayofweek

    df['tgt_hour_float'] = tgt_hour
    df['tgt_dow']        = tgt_dow
    df['tgt_hour_sin']   = np.sin(2 * np.pi * tgt_hour / 24)
    df['tgt_hour_cos']   = np.cos(2 * np.pi * tgt_hour / 24)
    df['tgt_dow_sin']    = np.sin(2 * np.pi * tgt_dow / 7)
    df['tgt_dow_cos']    = np.cos(2 * np.pi * tgt_dow / 7)

    # Конвейерный сигнал: status_3 для ближних горизонтов, status_1 для дальних
    df['status3_x_horizon'] = df['status_3_at_ref'] * h
    df['pipeline_signal']   = np.where(
        h <= 2, df['status_3_at_ref'],
        np.where(h <= 4, df['status_2_at_ref'], df['status_1_at_ref'])
    )
    return df


_TGT_LOOKUP_COLS = ['route_id', 'timestamp', 'target_1h']

def build_train_matrix(
    feat_df: pd.DataFrame,
    train_full: pd.DataFrame,
    subsample_n: int = 4,
) -> pd.DataFrame:
    """
    Построить обучающую матрицу: subsampled ref timestamps × 8 горизонтов.

    feat_df      — DataFrame с признаками для всех reference timestamps
    train_full   — полный train (для lookup целевых значений по tgt_ts)
    subsample_n  — шаг субсэмплинга (каждый N-й timestamp внутри маршрута)
    """
    feat_df = feat_df.sort_values(['route_id', 'timestamp']).copy()
    feat_df['_rn'] = feat_df.groupby('route_id').cumcount()
    feat_sub = feat_df[feat_df['_rn'] % subsample_n == 0].drop('_rn', axis=1).copy()

    tgt_lookup = (
        train_full[_TGT_LOOKUP_COLS]
        .rename(columns={'timestamp': 'tgt_ts', 'target_1h': 'target'})
    )

    dfs = []
    for h in HORIZONS:
        df_h = add_horizon_features(feat_sub, h)
        df_h = df_h.merge(tgt_lookup, on=['route_id', 'tgt_ts'], how='left')
        dfs.append(df_h)

    result = pd.concat(dfs, ignore_index=True)
    result = result.dropna(subset=['target'])
    return result


def build_inference_matrix(feat_df: pd.DataFrame, cutoff: pd.Timestamp) -> pd.DataFrame:
    """
    Построить матрицу для инференса: 8 горизонтов × 1000 маршрутов.
    Использует признаки только в момент cutoff (без будущих данных).
    """
    cutoff_df = feat_df[feat_df['timestamp'] == cutoff].copy()
    assert len(cutoff_df) == 1000, f"Ожидается 1000 маршрутов в cutoff, получено {len(cutoff_df)}"

    dfs = [add_horizon_features(cutoff_df, h) for h in HORIZONS]
    return pd.concat(dfs, ignore_index=True)


print("Функции feature engineering определены.")

In [ ]:
FEATURE_COLS = [
    # Маршрут (категориальный)
    'route_id',
    # Горизонт прогноза
    'horizon',
    # Маршрутные статистики
    'route_mean', 'route_median', 'route_std',
    'route_q25', 'route_q75', 'route_zero_frac', 'route_cv',
    # Временные признаки целевого момента
    'tgt_hour_float', 'tgt_dow',
    'tgt_hour_sin', 'tgt_hour_cos', 'tgt_dow_sin', 'tgt_dow_cos',
    # Временные признаки точки отсчёта
    'ref_hour', 'ref_dow',
    # Лаги таргета
    'target_lag_1', 'target_lag_2', 'target_lag_3', 'target_lag_4',
    'target_lag_6', 'target_lag_12', 'target_lag_24',
    'target_lag_48', 'target_lag_96', 'target_lag_336',
    # Rolling mean таргета
    'target_roll_mean_2h', 'target_roll_mean_6h', 'target_roll_mean_12h',
    'target_roll_mean_24h', 'target_roll_mean_7d',
    # Rolling std таргета
    'target_roll_std_6h', 'target_roll_std_24h',
    # EWM
    'target_ewm_span4',
    # Разности
    'target_diff_1', 'target_diff_48',
    # ── КОНВЕЙЕРНЫЕ ФИЧИ (ключевое новшество Эксп. 3) ──────────────────────
    # Status в момент reference
    'status_1_at_ref', 'status_2_at_ref', 'status_3_at_ref',
    'status_4_at_ref', 'status_5_at_ref',
    # Лаги status_3 (pipeline look-back)
    'status_3_lag_1', 'status_3_lag_2', 'status_3_lag_3', 'status_3_lag_4',
    # Rolling mean status
    'status_1_roll_mean_2h', 'status_2_roll_mean_2h', 'status_3_roll_mean_2h',
    'status_4_roll_mean_24h', 'status_5_roll_mean_24h',
    # Взаимодействия конвейер × горизонт
    'status3_x_horizon', 'pipeline_signal',
]

CAT_FEATURES = ['route_id']

print(f"Всего признаков: {len(FEATURE_COLS)}")
print(f"Категориальных: {CAT_FEATURES}")
print(f"\nКонвейерных фичей: status_*_at_ref (5) + status_3_lag_* (4) + rolling (5) + interactions (2) = 16")

## Кросс-валидация (rolling-origin по субботам)

Схема валидации **единая для всех экспериментов**:
- 3 фолда: cutoff = Oct 11 / Oct 18 / Oct 25 в 10:30
- Для каждого фолда: обучение на данных до cutoff, прогноз на 8 шагов (11:00–14:30)
- Постобработка: `clamp(0)` + калибровка через `ratio = Σy_val / Σŷ_val` (обнуляет RBias)
- Метрика: **WAPE + |Relative Bias|**

In [ ]:
fold_results    = []
last_model      = None
all_horizon_acc = {h: {'WAPE': [], 'Total': []} for h in HORIZONS}

for fold_idx, cutoff in enumerate(VAL_CUTOFFS):
    t0 = time.time()
    print(f"\n{'='*60}")
    print(f"Фолд {fold_idx+1}/{len(VAL_CUTOFFS)}: cutoff = {cutoff}")

    # 1. Данные до cutoff (имитация условий теста)
    train_cut = train[train['timestamp'] <= cutoff].copy()
    print(f"  Строк в train_cut: {len(train_cut):,}")

    # 2. Маршрутные статистики из обучающих данных
    route_stats = compute_route_stats(train_cut)

    # 3. Признаки для всех reference timestamps
    print("  Вычисляю ref_features ...", end=' ', flush=True)
    feat_df = compute_ref_features(train_cut, route_stats)
    print("готово")

    # 4. Обучающая матрица (субсэмплинг + 8 горизонтов)
    print("  Строю train_matrix ...", end=' ', flush=True)
    train_matrix = build_train_matrix(feat_df, train, subsample_n=SUBSAMPLE_N)
    print(f"готово — {len(train_matrix):,} строк")

    X_tr = train_matrix[FEATURE_COLS]
    y_tr = train_matrix['target']

    # 5. Обучение LightGBM
    print("  Обучаю LightGBM ...", end=' ', flush=True)
    model = lgb.LGBMRegressor(**LGBM_PARAMS)
    model.fit(X_tr, y_tr, categorical_feature=CAT_FEATURES)
    print("готово")

    # 6. Инференс: признаки в момент cutoff → прогноз на 8 горизонтов
    infer_df = build_inference_matrix(feat_df, cutoff)
    infer_df = infer_df.sort_values(['horizon', 'route_id']).reset_index(drop=True)

    preds_raw = np.maximum(model.predict(infer_df[FEATURE_COLS]), 0)

    # 7. Реальные значения таргета (8 шагов после cutoff)
    actual_rows = []
    for h in HORIZONS:
        act = train[train['timestamp'] == cutoff + h * STEP][['route_id', 'target_1h']].copy()
        act['horizon'] = h
        actual_rows.append(act)
    actual_df = (
        pd.concat(actual_rows)
        .sort_values(['horizon', 'route_id'])
        .reset_index(drop=True)
    )

    y_val = actual_df['target_1h'].values

    # 8. Калибровка (ratio обнуляет Relative Bias)
    ratio    = y_val.sum() / preds_raw.sum()
    preds_cal = preds_raw * ratio

    # 9. Метрики фолда
    wape_v  = wape(y_val, preds_cal)
    rbias_v = relative_bias(y_val, preds_cal)
    total_v = wape_v + abs(rbias_v)

    # 10. Метрики по горизонтам
    horizon_metrics = []
    for h in HORIZONS:
        mask = (actual_df['horizon'] == h).values
        y_h  = y_val[mask]
        p_h  = preds_cal[mask]
        w_h  = wape(y_h, p_h)
        b_h  = relative_bias(y_h, p_h)
        t_h  = w_h + abs(b_h)
        horizon_metrics.append({'horizon': h, 'WAPE': w_h, 'RBias': b_h, 'Total': t_h})
        all_horizon_acc[h]['WAPE'].append(w_h)
        all_horizon_acc[h]['Total'].append(t_h)

    elapsed = time.time() - t0
    print(f"  WAPE={wape_v:.4f}  |RBias|={abs(rbias_v):.4f}  Total={total_v:.4f}")
    print(f"  ratio={ratio:.4f}  время={elapsed:.1f}с")

    fold_results.append({
        'fold': fold_idx + 1,
        'cutoff': str(cutoff.date()),
        'WAPE': wape_v,
        'RBias': rbias_v,
        'Total': total_v,
        'ratio': ratio,
        'horizon_metrics': horizon_metrics,
        'elapsed': elapsed,
    })
    last_model = model

print(f"\n{'='*60}")
print("Валидация завершена.")

In [ ]:
# Сводная таблица результатов
rows = [
    {
        'Фолд': r['fold'],
        'Cutoff': r['cutoff'],
        'WAPE': r['WAPE'],
        '|RBias|': abs(r['RBias']),
        'Total': r['Total'],
        'Ratio': r['ratio'],
        'Время, с': round(r['elapsed']),
    }
    for r in fold_results
]
res_df = pd.DataFrame(rows)

mean_row = res_df[['WAPE','|RBias|','Total','Ratio','Время, с']].mean().to_dict()
mean_row.update({'Фолд': 'Среднее', 'Cutoff': ''})
res_df = pd.concat([res_df, pd.DataFrame([mean_row])], ignore_index=True)

print("Результаты кросс-валидации:")
print(res_df.to_string(index=False, float_format='{:.4f}'.format))

mean_total = float(res_df.loc[res_df['Фолд'] == 'Среднее', 'Total'].values[0])
print(f"\nБейзлайн: {BASELINE:.4f}  →  Эксп 3: {mean_total:.4f}  (Δ = {mean_total-BASELINE:+.4f})")

In [ ]:
# Ошибка по горизонтам (среднее по 3 фолдам)
h_wape  = [np.mean(all_horizon_acc[h]['WAPE'])  for h in HORIZONS]
h_total = [np.mean(all_horizon_acc[h]['Total']) for h in HORIZONS]
h_labels = [f'+{h*30}м' for h in HORIZONS]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый: столбчатый — WAPE и Total по горизонту
ax = axes[0]
x = np.arange(len(HORIZONS))
ax.bar(x - 0.2, h_total, 0.35, label='Total', color='steelblue', alpha=0.85)
ax.bar(x + 0.2, h_wape,  0.35, label='WAPE',  color='dodgerblue', alpha=0.85)
ax.axhline(BASELINE, color='red', linestyle='--', linewidth=1.5, label=f'Бейзлайн ({BASELINE})')
ax.set_xticks(x)
ax.set_xticklabels(h_labels)
ax.set_title('Метрика по горизонтам прогноза', fontsize=12)
ax.set_ylabel('Метрика')
ax.set_xlabel('Горизонт (минут вперёд)')
ax.legend()

# Правый: метрики по фолдам
ax = axes[1]
for r in fold_results:
    totals = [hm['Total'] for hm in r['horizon_metrics']]
    ax.plot(HORIZONS, totals, 'o-', alpha=0.7, label=f"Фолд {r['fold']} ({r['cutoff']})")
ax.plot(HORIZONS, h_total, 'k^--', linewidth=2, markersize=8, label='Среднее')
ax.axhline(BASELINE, color='red', linestyle='--', linewidth=1.5)
ax.set_xticks(HORIZONS)
ax.set_xticklabels(h_labels)
ax.set_title('Total по горизонтам (все фолды)', fontsize=12)
ax.set_ylabel('WAPE + |RBias|')
ax.set_xlabel('Горизонт')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Таблица
horizon_df = pd.DataFrame({
    'Горизонт': [f'h={h} (+{h*30}мин)' for h in HORIZONS],
    'WAPE':  [f'{v:.4f}' for v in h_wape],
    'Total': [f'{v:.4f}' for v in h_total],
})
print(horizon_df.to_string(index=False))

## Feature Importance

Анализируем важность признаков по последнему фолду. Ожидаем, что `status_3_at_ref`, `pipeline_signal` и `route_mean` окажутся в топе.

In [ ]:
fi_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': last_model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Топ-25 признаков
ax = axes[0]
top = fi_df.head(25)
ax.barh(top['feature'][::-1], top['importance'][::-1], color='steelblue', alpha=0.85)
ax.set_title('Топ-25 признаков по важности (split gain)', fontsize=12)
ax.set_xlabel('Importance')
plt.setp(ax.get_yticklabels(), fontsize=9)

# Важность конвейерных фичей отдельно
ax = axes[1]
conveyor_names = [f for f in FEATURE_COLS if 'status' in f or 'pipeline' in f]
conv_fi = fi_df[fi_df['feature'].isin(conveyor_names)].sort_values('importance', ascending=False)
colors = ['tomato' if 'status_3' in n or 'pipeline' in n else 'steelblue' for n in conv_fi['feature']]
ax.barh(conv_fi['feature'][::-1], conv_fi['importance'][::-1], color=colors[::-1], alpha=0.85)
ax.set_title('Важность конвейерных признаков\n(красный = status_3 / pipeline)', fontsize=12)
ax.set_xlabel('Importance')
plt.setp(ax.get_yticklabels(), fontsize=9)

plt.tight_layout()
plt.show()

print("Топ-15 признаков:")
print(fi_df.head(15)[['feature', 'importance']].to_string(index=False))

zero_imp = fi_df[fi_df['importance'] == 0]['feature'].tolist()
if zero_imp:
    print(f"\nПризнаки с нулевой важностью: {zero_imp}")
else:
    print("\nПризнаков с нулевой важностью нет.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Финальная модель (на полном train) + Тестовые прогнозы
# ══════════════════════════════════════════════════════════════════════════════
print("Обучаю финальную модель на полном train ...")
t0 = time.time()

route_stats_full = compute_route_stats(train)

print("  Вычисляю ref_features (полный train) ...", end=' ', flush=True)
feat_full = compute_ref_features(train, route_stats_full)
print("готово")

print("  Строю train_matrix ...", end=' ', flush=True)
train_matrix_full = build_train_matrix(feat_full, train, subsample_n=SUBSAMPLE_N)
print(f"готово — {len(train_matrix_full):,} строк")

print("  Обучаю LightGBM ...", end=' ', flush=True)
model_final = lgb.LGBMRegressor(**LGBM_PARAMS)
model_final.fit(
    train_matrix_full[FEATURE_COLS],
    train_matrix_full['target'],
    categorical_feature=CAT_FEATURES,
)
print(f"готово ({time.time()-t0:.1f}с)")

# ── Тестовые прогнозы ──────────────────────────────────────────────────────
print("\nГенерирую тестовые прогнозы ...")
infer_test = build_inference_matrix(feat_full, TEST_CUTOFF)
infer_test = infer_test.sort_values(['route_id', 'horizon']).reset_index(drop=True)

preds_test_raw = np.maximum(model_final.predict(infer_test[FEATURE_COLS]), 0)

# Калибровка: среднее ratio из валидационных фолдов
mean_ratio = np.mean([r['ratio'] for r in fold_results])
preds_test_cal = preds_test_raw * mean_ratio
print(f"  Калибровочный коэф (среднее по фолдам): {mean_ratio:.4f}")
print(f"  Прогнозы: min={preds_test_cal.min():.0f}  max={preds_test_cal.max():.0f}  mean={preds_test_cal.mean():.0f}")

# ── Формирование submission ─────────────────────────────────────────────────
sub = infer_test[['route_id', 'tgt_ts']].copy()
sub['y_pred'] = preds_test_cal          # float, без округления
sub = sub.rename(columns={'tgt_ts': 'timestamp'})

test_ids = test[['id', 'route_id', 'timestamp']].copy()
sub = sub.merge(test_ids, on=['route_id', 'timestamp'], how='left')
sub = sub[['id', 'y_pred']].sort_values('id').reset_index(drop=True)

assert len(sub) == len(test), f"Ожидается {len(test)} строк, получено {len(sub)}"
assert sub['id'].notna().all(), "Есть незаматченные строки (NaN в id)"

out_path = '/Users/melikhovartem/Desktop/ИЗИ 200к/submission_exp03.csv'
sub.to_csv(out_path, index=False)
print(f"\nSubmission сохранён: {out_path}")
print(sub.head(10).to_string(index=False))

## Итоговые метрики

Сводная таблица для сравнения экспериментов между собой. **Копируйте значение из строки «Эксп 3» при сравнении.**

In [ ]:
mean_wape  = np.mean([r['WAPE']         for r in fold_results])
mean_rbias = np.mean([abs(r['RBias'])   for r in fold_results])
mean_total = np.mean([r['Total']        for r in fold_results])

print("=" * 65)
print("ИТОГОВЫЕ МЕТРИКИ — ЭКСПЕРИМЕНТ 3")
print("LightGBM + конвейерные фичи (одна модель, horizon как фича)")
print("=" * 65)
print(f"\nВалидация: rolling-origin, 3 субботних фолда")
print(f"           (Oct 11, Oct 18, Oct 25 @ 10:30 → прогноз 11:00–14:30)\n")

summary_df = pd.DataFrame({
    'Метрика':              ['WAPE', '|Relative Bias|', 'Total (WAPE+|RBias|)'],
    'Среднее по 3 фолдам': [f'{mean_wape:.4f}', f'{mean_rbias:.4f}', f'{mean_total:.4f}'],
})
print(summary_df.to_string(index=False))

# ── Сравнение с бейзлайнами ────────────────────────────────────────────────
print(f"\n{'─'*65}")
print("Сравнение с предыдущими экспериментами:")
comparison = pd.DataFrame([
    {'Метод': 'Последнее значение (Exp 0)',    'Total': 0.516,       'Δ vs baseline': '+0.134'},
    {'Метод': 'Среднее по маршруту (Exp 0)',   'Total': 0.382,       'Δ vs baseline': '0.000 (baseline)'},
    {'Метод': '▶ LightGBM+conveyor (Exp 3)',   'Total': mean_total,  'Δ vs baseline': f'{mean_total-0.382:+.4f}'},
])
print(comparison.to_string(index=False, float_format='{:.4f}'.format))

# ── Итоговый график ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
fold_labels = [r['cutoff'] for r in fold_results]
fold_totals = [r['Total']  for r in fold_results]
bars = ax.bar(fold_labels, fold_totals, color='steelblue', alpha=0.85, width=0.5)
ax.axhline(mean_total, color='navy', linewidth=2, label=f'Среднее = {mean_total:.4f}')
ax.axhline(BASELINE,   color='red',  linewidth=1.5, linestyle='--', label=f'Baseline = {BASELINE}')
for bar, v in zip(bars, fold_totals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.4f}',
            ha='center', va='bottom', fontsize=10)
ax.set_title('Total metric по фолдам', fontsize=12)
ax.set_ylabel('WAPE + |RBias|')
ax.legend()
ax.set_ylim(0, max(fold_totals) * 1.15)

ax = axes[1]
ax.plot(HORIZONS, h_total, 'o-', color='steelblue', linewidth=2, markersize=7, label='Total (Exp 3)')
ax.plot(HORIZONS, h_wape,  's--', color='dodgerblue', linewidth=1.5, markersize=6, label='WAPE (Exp 3)')
ax.axhline(BASELINE, color='red', linestyle='--', linewidth=1.5, label=f'Baseline = {BASELINE}')
ax.axhline(mean_total, color='navy', linestyle=':', linewidth=1.5, label=f'Ср. Total = {mean_total:.4f}')
ax.set_xticks(HORIZONS)
ax.set_xticklabels([f'+{h*30}м' for h in HORIZONS])
ax.set_title('Total по горизонтам прогноза', fontsize=12)
ax.set_ylabel('Метрика')
ax.set_xlabel('Горизонт')
ax.legend(fontsize=9)

plt.suptitle('Эксперимент 3: LightGBM + конвейерные фичи', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f"\n{'='*65}")
print(f"  ИТОГ: Total = {mean_total:.4f}  (бейзлайн: {BASELINE}  Δ={mean_total-BASELINE:+.4f})")
print(f"{'='*65}")